# NB02 · Dependency discovery + cascade engine — online discrete-time multivariate Hawkes  (CPU · ~15–40 min)

Trains the **pooled Hawkes prior** that powers the cascade engine, and measures everything that matters for it.

* Point-process GLM: `λ_{z,i}(t) = μ_i + Σ_{j,b} A[i,j,b]·(C_b[j](z,t) + s_j·Σ_{z'}W_{zz'}C_b[j](z',t))`, A ≥ 0 (softplus), exponential lag basis τ = 2/8/24/72 h, **strictly causal**.
* L1 + per-edge group lasso; masks only **forbid** physically impossible edges (→WX, →SEI), never assert any.
* Implemented in **NumPy with analytic gradients** (no GPU needed; a finite-difference gradient check is run below).
* **Metrics:** held-out NLL / information gain vs constant-rate, edge AUROC/AP/F1 vs hidden `G_true`, lag error, cascade reproduction number ρ, edge sensitivity ∂ρ/∂A, feedback-loop gain, edge evidence cards (bootstrap CI, likelihood-ratio, precedence-vs-shuffle), baselines (lagged correlation, shuffled-timing), unseen-city transfer, **online adaptation under injected drift** (prequential NLL, CUSUM change-point detection delay).

**Inputs:** NB00 output.  **Outputs:** `hawkes_prior.npz`, `hawkes_report.json`, `figs/`.  Checkpoints each λ-grid point → resumable.

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
QUICK = False
CFG = dict(grid=[(2e-5, 2e-5), (1e-4, 1e-4), (5e-4, 5e-4)], epochs=25, bs=24, lr=0.05, n_boot=10, n_edges_cards=12, n_null=2)
if QUICK: CFG.update(grid=[(1e-4, 1e-4)], epochs=3, n_boot=2, n_edges_cards=3, n_null=1)
MAN = pd.read_csv(find('w72_manifest.csv')); ROOT = os.path.dirname(find('w72_manifest.csv'))
CITY = {'anchor': W.make_city('anchor'), 'heldout': W.make_city('heldout')}
I = J = W.NTYPE; Bn = len(W.TAUS); MASK = W.forbidden_type_mask()[:, :, None]; PM = W.eval_pair_mask(); G = W.g_true_matrix()
print(MAN.groupby('split').size().to_dict())

In [ ]:
# ---------------- events from the OBSERVED stream (what a deployed system would see, faults included)
def prep(row, profile=None):
    d = W.load_episode(f'{ROOT}/{row.path}'); asof, stale, info = W.observe(d['X'], profile or row.profile, int(row.seed) + 777)
    lv = W.levels_from_stress(W.stress_from_vars(W.fill_defaults(asof))); N = W.events_from_levels(lv)
    return dict(id=row.ep_id, hazard=row.hazard, city=row.city, N=N, C=W.lag_conv(N), W=CITY[row.city]['W'], drift_at=row.drift_at, meta=d['meta'])
def load_split(split, n=None):
    rows = MAN[MAN.split == split]; rows = rows.head(n) if n else rows
    return [prep(r) for r in tqdm(list(rows.itertuples()), desc=f'events {split}')]
DS = {s: load_split(s) for s in ['train', 'val', 'test', 'heldout', 'drift']}
ev = np.concatenate([e['N'].reshape(-1, J) for e in DS['train']]).sum(0)
print(pd.Series(ev, index=W.EVENT_TYPES).astype(int).to_dict())

In [ ]:
# ---------------- model utilities
def inv_softplus(y): y = np.maximum(y, 1e-6); return np.log(np.expm1(y))
def stack_rows(eps):
    Cr, WCr = W._rows([e['C'] for e in eps], eps[0]['W']); Nr = np.concatenate([e['N'].reshape(-1, J) for e in eps]).astype(np.float32); return Cr, WCr, Nr
def layer_scores(A): return W.layer_matrix(A)
def edge_metrics(A, thr=None):
    from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
    M = layer_scores(A); y = G[PM]; s = M[PM]; out = dict(auroc=roc_auc_score(y, s), ap=average_precision_score(y, s))
    if thr is None:
        p, r, t = precision_recall_curve(y, s); f = 2 * p * r / np.maximum(p + r, 1e-9); k = int(np.nanargmax(f[:-1])); thr = float(t[k])
    pred = s >= thr; tp = (pred & y).sum(); out.update(f1=2 * tp / max(pred.sum() + y.sum(), 1), thr=thr, precision=tp / max(pred.sum(), 1), recall=tp / y.sum())
    return out
def fit_hawkes(eps, l1, lg, epochs=20, bs=24, lr=0.05, init=None, l2_prior=0.0, val=None, tag='fit', seed=0, mask=MASK, desc=True):
    rng = np.random.default_rng(seed); Cv = None
    if val is not None: Cv = stack_rows(val)
    if init is None:
        r0 = np.concatenate([e['N'].reshape(-1, J) for e in eps[:50]]).mean(0)
        rawA = np.full((I, J, Bn), inv_softplus(0.002)); rawmu = inv_softplus(np.maximum(r0 * 0.5, 1e-4)); raws = np.zeros(J); A0 = None
    else:
        rawA = inv_softplus(init['A']); rawmu = inv_softplus(init['mu']); raws = np.log(np.clip(init['s'], 1e-3, 1 - 1e-3) / (1 - np.clip(init['s'], 1e-3, 1 - 1e-3))); A0 = init['A'].copy()
    opt = W.Adam([rawA.shape, rawmu.shape, raws.shape], lr=lr); hist = []
    for ep in tqdm(range(epochs), desc=tag, disable=not desc):
        perm = rng.permutation(len(eps)); tr = []
        for b in range(0, len(eps), bs):
            Cr, WCr, Nr = stack_rows([eps[k] for k in perm[b:b + bs]])
            A = W.softplus(rawA) * mask; mu = W.softplus(rawmu); s = W.sigmoid(raws)
            nll, dA, dmu, ds = W.hawkes_nll_grad(A, mu, s, Cr, WCr, Nr); _, rdA = W.hawkes_reg(A, l1, lg); dA = dA + rdA
            if A0 is not None and l2_prior > 0: dA = dA + 2 * l2_prior * (A - A0)
            opt.step([rawA, rawmu, raws], [dA * mask * W.sigmoid(rawA), dmu * W.sigmoid(rawmu), ds * s * (1 - s)]); tr.append(nll)
        A = W.softplus(rawA) * mask; mu = W.softplus(rawmu); s = W.sigmoid(raws); h = dict(epoch=ep, train_nll=float(np.mean(tr)))
        if Cv is not None: h['val_nll'] = W.hawkes_nll_grad(A, mu, s, Cv[0], Cv[1], Cv[2], need_grad=False); h['val_auroc'] = edge_metrics(A)['auroc']; h['rho'] = W.branching_ratio(A)[0]
        hist.append(h)
    return dict(A=A, mu=mu, s=s, taus=W.TAUS), pd.DataFrame(hist)
def nll_on(P, eps):
    Cr, WCr, Nr = stack_rows(eps); return W.hawkes_nll_grad(P['A'], P['mu'], P['s'], Cr, WCr, Nr, need_grad=False)
def const_rate_nll(train_eps, eps):
    r = np.maximum(np.concatenate([e['N'].reshape(-1, J) for e in train_eps]).mean(0), 1e-8); Nr = np.concatenate([e['N'].reshape(-1, J) for e in eps])
    return float((r - Nr * np.log(r)).sum(1).mean())

In [ ]:
# ---------------- 1) gradient check (analytic vs finite differences) -- proves the hand-written maths
sub = DS['train'][:2]; Cr, WCr, Nr = stack_rows(sub); Cr, WCr, Nr = Cr[:3000], WCr[:3000], Nr[:3000]
rng = np.random.default_rng(0); A = rng.uniform(0.001, 0.03, (I, J, Bn)); mu = rng.uniform(0.002, 0.02, I); s = rng.uniform(0.1, .8, J)
f, dA, dmu, ds = W.hawkes_nll_grad(A, mu, s, Cr, WCr, Nr); errs = []
for (i, j, b) in [(3, 5, 1), (7, 2, 0), (12, 11, 3)]:
    A2 = A.copy(); A2[i, j, b] += 1e-6; errs.append(abs((W.hawkes_nll_grad(A2, mu, s, Cr, WCr, Nr, need_grad=False) - f) / 1e-6 - dA[i, j, b]))
s2 = s.copy(); s2[4] += 1e-6; errs.append(abs((W.hawkes_nll_grad(A, mu, s2, Cr, WCr, Nr, need_grad=False) - f) / 1e-6 - ds[4]))
mu2 = mu.copy(); mu2[6] += 1e-6; errs.append(abs((W.hawkes_nll_grad(A, mu2, s, Cr, WCr, Nr, need_grad=False) - f) / 1e-6 - dmu[6]))
print('max |analytic - finite-diff| =', max(errs)); assert max(errs) < 1e-3

## Pooled prior: λ-grid, selection on validation NLL (checkpointed)

In [ ]:
results = {}
for (l1, lg) in CFG['grid']:
    ck = f'{OUT}/ck_hawkes_{l1:g}_{lg:g}.pkl'
    if os.path.exists(ck): results[(l1, lg)] = pickle.load(open(ck, 'rb')); print('resumed', l1, lg); continue
    P, hist = fit_hawkes(DS['train'], l1, lg, epochs=CFG['epochs'], bs=CFG['bs'], lr=CFG['lr'], val=DS['val'], tag=f'hawkes l1={l1:g}')
    results[(l1, lg)] = dict(P=P, hist=hist); pickle.dump(results[(l1, lg)], open(ck, 'wb'))
best = min(results, key=lambda k: results[k]['hist'].val_nll.iloc[-1]); PRIOR = results[best]['P']
print('selected (l1, lg) =', best)
base_val = const_rate_nll(DS['train'], DS['val']); print(f'val NLL: constant-rate {base_val:.4f} | Hawkes {results[best]["hist"].val_nll.iloc[-1]:.4f} | information gain per row {base_val - results[best]["hist"].val_nll.iloc[-1]:.4f} nats')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for k, r in results.items():
    h = r['hist']; ax[0].plot(h.epoch, h.val_nll, label=f'val l1={k[0]:g}'); ax[0].plot(h.epoch, h.train_nll, ls=':', c=ax[0].lines[-1].get_color()); ax[1].plot(h.epoch, h.val_auroc, label=f'l1={k[0]:g}'); ax[2].plot(h.epoch, h.rho)
ax[0].axhline(base_val, c='k', ls='--', label='constant rate'); ax[0].set_title('Poisson NLL / row'); ax[0].legend(fontsize=7); ax[0].set_xlabel('epoch')
ax[1].set_title('edge AUROC vs G_true (val)'); ax[1].set_xlabel('epoch'); ax[2].set_title('branching ratio ρ'); ax[2].axhline(1, c='r', ls='--'); ax[2].set_xlabel('epoch'); savefig('10_hawkes_training')

## Dependency discovery vs ground truth — with baselines

In [ ]:
# baselines: (a) lagged correlation graph, (b) shuffled-timing Hawkes (null), (c) true prior
def lagcorr_scores(eps, maxlag=24):
    Ls = [np.stack([e['N'][..., 2 * l].astype(np.float32) + e['N'][..., 2 * l + 1] for l in range(10)], -1) for e in eps]
    allx = np.concatenate([x.reshape(-1, 10) for x in Ls]); m, sd = allx.mean(0), allx.std(0) + 1e-6; best = np.zeros((10, 10)) - 1
    for lag in range(1, maxlag + 1):
        cov = np.zeros((10, 10)); n = 0
        for x in Ls:
            a = ((x[:-lag] - m) / sd).reshape(-1, 10); b = ((x[lag:] - m) / sd).reshape(-1, 10); cov += a.T @ b; n += len(a)
        best = np.maximum(best, cov / n)
    return best                                                    # [from, to]
def auc_of(M):
    from sklearn.metrics import roc_auc_score, average_precision_score
    return roc_auc_score(G[PM], M[PM]), average_precision_score(G[PM], M[PM])
rng = np.random.default_rng(1)
def shuffled(eps):
    out = []
    for e in eps:
        N = e['N'].copy()
        for j in range(J): N[:, :, j] = np.roll(N[:, :, j], int(rng.integers(5, N.shape[0] - 5)), axis=0)
        out.append(dict(e, N=N, C=W.lag_conv(N)))
    return out
sh_eps = shuffled(DS['train'][:min(len(DS['train']), 200)]); P_sh, _ = fit_hawkes(sh_eps, *best, epochs=max(3, CFG['epochs'] // 2), tag='circular-shift null', desc=False)
def perm_null(eps, seed):
    # permute each event type's whole episode among episodes of the SAME hazard: keeps hazard-level co-activity (storms happen in similar windows) but destroys within-episode lag structure
    rg = np.random.default_rng(seed); haz = np.array([e['hazard'] for e in eps]); Ns = [e['N'].copy() for e in eps]
    for h in np.unique(haz):
        idx = np.where(haz == h)[0]
        for j in range(J):
            for a, b in zip(idx, rg.permutation(idx)): Ns[a][:, :, j] = eps[b]['N'][:, :, j]
    return [dict(e, N=N, C=W.lag_conv(N)) for e, N in zip(eps, Ns)]
base_eps = DS['train'][:min(len(DS['train']), 250)]; M_null = []
for k in range(CFG['n_null']):
    P_pn, _ = fit_hawkes(perm_null(base_eps, 10 + k), *best, epochs=CFG['epochs'] if not QUICK else 2, tag=f'permuted null {k}', desc=False); M_null.append(layer_scores(P_pn['A']))
M_null = np.mean(M_null, 0); M_main = layer_scores(PRIOR['A']); M_corr = M_main - M_null
tab = []
for name, M in [('Hawkes prior (ours), raw score', M_main), ('Hawkes, null-corrected (raw - episode-permuted null)', M_corr), ('lagged correlation', lagcorr_scores(DS['train'])), ('Hawkes on circular-shifted events (null)', layer_scores(P_sh['A'])), ('Hawkes on episode-permuted events (null)', M_null), ('random', np.random.default_rng(0).random((10, 10)))]:
    a, p = auc_of(M); tab.append(dict(method=name, AUROC=a, AP=p))
TAB = pd.DataFrame(tab).round(3); display(TAB)
print('Reading guide: the null rows show how much edge "evidence" comes from hazard-level co-activity alone (everything happens during the same storm window). The null-corrected score keeps only what the within-episode lag structure adds.')
em = edge_metrics(PRIOR['A']); em_c = None
print({k: round(float(v), 3) for k, v in em.items()}, '  <- best-F1 (ORACLE threshold chosen against G_true: an upper bound, not a deployable number)')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5)); M = layer_scores(PRIOR['A']); Mp = np.where(PM, M, np.nan)
for a_, (mat, ttl) in zip(ax[:2], [(G.astype(float), 'ground truth G_true'), (Mp / np.nanmax(Mp), 'learned integrated influence (normalised)')]):
    im = a_.imshow(mat, cmap='Blues'); a_.grid(False); a_.set_xticks(range(10)); a_.set_xticklabels(W.LAYERS, rotation=45); a_.set_yticks(range(10)); a_.set_yticklabels(W.LAYERS); a_.set_title(ttl); a_.set_xlabel('TO'); a_.set_ylabel('FROM')
Lg = W.layer_lag(PRIOR['A']); tl, el = [], []
for a, b, lo, hi, _ in W.GEDGES:
    if PM[W.LI[a], W.LI[b]] and M[W.LI[a], W.LI[b]] >= em['thr']: tl.append((lo + hi) / 2); el.append(Lg[W.LI[a], W.LI[b]])
ax[2].scatter(tl, el, c='tab:green'); mx = max(tl + el + [10]); ax[2].plot([0, mx], [0, mx], 'k--'); ax[2].set_xlabel('true lag window midpoint (h)'); ax[2].set_ylabel('estimated kernel-mean lag (h)')
lag_mae = float(np.mean(np.abs(np.array(tl) - np.array(el)))) if tl else float('nan'); ax[2].set_title(f'lag recovery on detected edges (MAE {lag_mae:.1f} h)'); savefig('11_edges_vs_truth')
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(6, 4))
for name, M_ in [('Hawkes raw', M_main), ('Hawkes null-corrected', M_corr), ('lagged corr', lagcorr_scores(DS['train'])), ('permuted null', M_null)]:
    p, r, _ = precision_recall_curve(G[PM], M_[PM]); ax.plot(r, p, label=name)
ax.set_xlabel('recall'); ax.set_ylabel('precision'); ax.legend(); ax.set_title('edge precision-recall'); savefig('12_edge_pr')

## Cascade reproduction number ρ, amplifying edges, feedback loops

In [ ]:
rho, Ai = W.branching_ratio(PRIOR['A']); rho_total = W.branching_ratio(PRIOR['A'], cross_only=False)[0]; ev_, evec = np.linalg.eig(Ai); k = int(np.argmax(np.abs(ev_))); v = np.real(evec[:, k]); v = v if v.sum() >= 0 else -v
ev2, evl = np.linalg.eig(Ai.T); k2 = int(np.argmax(np.abs(ev2))); u = np.real(evl[:, k2]); u = u if u.sum() >= 0 else -u
sens = np.outer(u, v) / max(float(u @ v), 1e-9)                    # d rho / d Ai[i,j]  (i=target type, j=source type)
sl = pd.DataFrame([dict(src=W.EVENT_TYPES[j], dst=W.EVENT_TYPES[i], dRho=sens[i, j] * (MASK[i, j, 0] > 0), weight=Ai[i, j]) for i in range(I) for j in range(J)]).sort_values('dRho', ascending=False)
print(f'cascade reproduction number (cross-system) rho = {rho:.3f}  | incl. same-layer deg<->fail coupling {rho_total:.3f}  (>=1 would be self-sustaining)'); display(sl.head(8).round(4))
# per-hazard rho: warm-started short refits per hazard
rho_h = {}
for hz in W.HAZARDS:
    es = [e for e in DS['train'] if e['hazard'] == hz]
    if len(es) < 3: continue
    P_h, _ = fit_hawkes(es, *best, epochs=3 if not QUICK else 1, init=PRIOR, l2_prior=1e-2, lr=0.02, desc=False, bs=len(es)); rho_h[hz] = W.branching_ratio(P_h['A'])[0]
# feedback loops (cycles in the layer graph) with loop gain = product of influences
import itertools
Ml = np.where(PM, layer_scores(PRIOR['A']), 0); thr = em['thr']; loops = []
for L in (2, 3):
    for cyc in itertools.permutations(range(10), L):
        if cyc[0] != min(cyc): continue
        w_ = [Ml[cyc[q], cyc[(q + 1) % L]] for q in range(L)]
        if min(w_) >= thr: loops.append(dict(loop=' → '.join(W.LAYERS[c] for c in cyc + (cyc[0],)), gain=float(np.prod(w_))))
LOOPS = pd.DataFrame(loops).sort_values('gain', ascending=False) if loops else pd.DataFrame(columns=['loop', 'gain']); display(LOOPS.head(6))
fig, ax = plt.subplots(1, 2, figsize=(13, 4)); ax[0].bar(list(rho_h), list(rho_h.values()), color='tab:purple'); ax[0].axhline(1, c='r', ls='--'); ax[0].set_title('cascade reproduction number ρ by hazard (warm-started refit)'); ax[0].tick_params(axis='x', rotation=30)
top = sl.head(10); ax[1].barh([f'{a}→{b}' for a, b in zip(top.src, top.dst)][::-1], top.dRho.values[::-1], color='tab:orange'); ax[1].set_title('most amplifying edges  ∂ρ/∂A'); savefig('13_rho_and_sensitivity')

## Edge evidence cards (what makes the graph credible)

In [ ]:
M = np.where(PM, layer_scores(PRIOR['A']), 0); top_pairs = [np.unravel_index(k, M.shape) for k in np.argsort(-M.ravel())[:CFG['n_edges_cards']]]
Cv = stack_rows(DS['val']); full_nll = W.hawkes_nll_grad(PRIOR['A'], PRIOR['mu'], PRIOR['s'], *Cv, need_grad=False)
def precedence(fr, to, win=12, eps=DS['val']):
    hit = base = tot = 0; rg = np.random.default_rng(3)
    for e in eps:
        src = (e['N'][..., 2 * fr] > 0); dst = np.where(e['N'][..., 2 * to] > 0); tot += len(dst[0])
        cs = np.cumsum(src, 0); lagc = np.zeros_like(cs); lagc[win:] = cs[:-win]; pre = (cs - lagc) > 0
        hit += pre[dst].sum(); sh = np.roll(src, int(rg.integers(24, 200)), axis=0); cs = np.cumsum(sh, 0); lagc = np.zeros_like(cs); lagc[win:] = cs[:-win]; base += ((cs - lagc) > 0)[dst].sum()
    return hit, base, tot
PRIOR_OK = lambda a, b: not (a == 'COM' or b in ('WX', 'SEI') or (a == 'HLT' and b not in ('HOS', 'EMS')))
mech = lambda a, b: 'cyber' if 'COM' in (a, b) else ('logical' if a in ('EMS', 'HOS', 'HLT') else 'physical')
cards = []; rg = np.random.default_rng(5)
for (fr, to) in tqdm(top_pairs, desc='evidence cards'):
    m2 = MASK.copy(); m2[2 * to:2 * to + 2, 2 * fr:2 * fr + 2, :] = 0
    P_wo, _ = fit_hawkes(DS['train'][:150], *best, epochs=2 if not QUICK else 1, init=PRIOR, lr=0.02, desc=False, mask=m2, bs=50)
    wo = W.hawkes_nll_grad(P_wo['A'], P_wo['mu'], P_wo['s'], *Cv, need_grad=False); lr_stat = 2 * len(Cv[0]) * (wo - full_nll)
    boots = []
    for b in range(CFG['n_boot']):
        idx = rg.integers(0, len(DS['train']), min(len(DS['train']), 120)); P_b, _ = fit_hawkes([DS['train'][k] for k in idx], *best, epochs=2 if not QUICK else 1, init=PRIOR, lr=0.02, desc=False, bs=60); boots.append(layer_scores(P_b['A'])[fr, to])
    h, bs_, tot = precedence(fr, to); a, b = W.LAYERS[fr], W.LAYERS[to]
    cards.append(dict(edge=f'{a}→{b}', weight=M[fr, to], lag_h=W.layer_lag(PRIOR['A'])[fr, to], ci_lo=np.percentile(boots, 5), ci_hi=np.percentile(boots, 95), LR_stat=lr_stat,
                      precedes_12h=f'{h}/{tot}', shuffled_baseline=f'{bs_}/{tot}', mechanism=mech(a, b), unsupported=not PRIOR_OK(a, b), in_G_true=bool(G[fr, to])))
CARDS = pd.DataFrame(cards); display(CARDS.round(3))

## Generalisation: unseen city, from-scratch discovery, cold-start curve

In [ ]:
ho = DS['heldout']; nll_zs = nll_on(PRIOR, ho); base_ho = const_rate_nll(DS['train'], ho)
print(f'held-out city NLL: constant-rate {base_ho:.4f} | pooled prior (zero-shot) {nll_zs:.4f}')
# cold start: adapt prior with k hours of data from the new city (k = number of episodes * 240 h / scaled) -- use first n episodes
cold = []
for n in ([0, 1, 2, 4, 8, 16, 32] if not QUICK else [0, 2]):
    n = min(n, len(ho) // 2); tr_, te_ = ho[:n], ho[len(ho) // 2:]
    P_c = PRIOR if n == 0 else fit_hawkes(tr_, *best, epochs=4 if not QUICK else 1, init=PRIOR, l2_prior=1e-2, lr=0.02, desc=False, bs=max(n, 1))[0]
    cold.append(dict(n_episodes=n, hours=n * 240, test_nll=nll_on(P_c, te_), edge_auroc=edge_metrics(P_c['A'])['auroc']))
COLD = pd.DataFrame(cold); display(COLD.round(4))
P_scr, _ = fit_hawkes(ho[:len(ho) // 2], *best, epochs=max(3, CFG['epochs'] // 2), desc=False, tag='scratch'); em_scr = edge_metrics(P_scr['A'], thr=em['thr'])
lc = auc_of(lagcorr_scores(ho[:len(ho) // 2])); print(f'edge discovery on UNSEEN city from scratch: Hawkes AUROC {em_scr["auroc"]:.3f}, F1@train-thr {em_scr["f1"]:.3f} | lagged-corr AUROC {lc[0]:.3f}')
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8)); ax[0].plot(COLD.hours, COLD.test_nll, 'o-'); ax[0].axhline(base_ho, c='k', ls='--', label='constant rate'); ax[0].set_xlabel('hours of new-city data seen'); ax[0].set_ylabel('held-out NLL'); ax[0].set_title('cold-start curve'); ax[0].legend()
ax[1].plot(COLD.hours, COLD.edge_auroc, 'o-', c='tab:green'); ax[1].set_xlabel('hours of new-city data seen'); ax[1].set_ylabel('edge AUROC'); ax[1].set_title('graph quality vs data seen'); savefig('14_cold_start')

## Online learning under concept drift
Paired flood episodes: same weather, but at t=100 the drainage capacity halves in the *drifted* run. The model streams the episode every 6 h, takes 5 Adam steps on the last 7 days with exponential forgetting (half-life 60 h) and an L2 pull toward the pooled prior. **Prequential NLL** (score the next 6 h *before* learning from them) compares online vs frozen prior; a one-sided CUSUM on the HYD→TRF weight is the change-point detector.

In [ ]:
def online_run(e, step=6, hl=60.0, win=168, n_steps=5, lr=0.01, l2=1e-2):
    T, Z = e['N'].shape[:2]; P = {k: (v.copy() if hasattr(v, 'copy') else v) for k, v in PRIOR.items()}; opt = None; rec = []
    rawA = inv_softplus(np.maximum(P['A'], 1e-4)); rawmu = inv_softplus(P['mu']); A0 = PRIOR['A']
    C = e['C']; WC = np.einsum('zy,tybj->tzbj', e['W'], C); N = e['N'].astype(np.float32)
    for tk in range(48, T - step, step):
        sl_ = slice(tk, tk + step); R = lambda s_: (C[s_].reshape(-1, Bn, J), WC[s_].reshape(-1, Bn, J), N[s_].reshape(-1, J))
        A = W.softplus(rawA) * MASK; mu = W.softplus(rawmu)
        nll_on_ = W.hawkes_nll_grad(A, mu, P['s'], *R(sl_), need_grad=False); nll_fz = W.hawkes_nll_grad(PRIOR['A'], PRIOR['mu'], PRIOR['s'], *R(sl_), need_grad=False)
        rec.append(dict(t=tk, nll_online=nll_on_, nll_frozen=nll_fz, edge=layer_scores(A)[W.LI['HYD'], W.LI['TRF']], rho=W.branching_ratio(A)[0]))
        lo = max(0, tk + step - win); past = slice(lo, tk + step); w = np.exp(-np.log(2) * (tk + step - 1 - np.arange(lo, tk + step)) / hl); w = np.repeat(w, Z).astype(np.float32)
        Cr, WCr, Nr = (x for x in (C[past].reshape(-1, Bn, J), WC[past].reshape(-1, Bn, J), N[past].reshape(-1, J)))
        opt = W.Adam([rawA.shape, rawmu.shape], lr=lr)
        for _ in range(n_steps):
            A = W.softplus(rawA) * MASK; mu = W.softplus(rawmu); _, dA, dmu, _ = W.hawkes_nll_grad(A, mu, P['s'], Cr, WCr, Nr, w=w)
            dA = dA + 2 * l2 * (A - A0); opt.step([rawA, rawmu], [dA * MASK * W.sigmoid(rawA), dmu * W.sigmoid(rawmu)])
    return pd.DataFrame(rec)
def cusum(x, base_n=5, k=0.5, h=5.0):
    x = np.asarray(x); mu_, sd = x[:base_n].mean(), max(x[:base_n].std(), 0.05 * abs(x[:base_n].mean()) + 1e-6); S = 0.0
    for i in range(base_n, len(x)):
        S = max(0.0, S + (x[i] - mu_) / sd - k)
        if S > h: return i
    return None
dr = DS['drift']; half = len(dr) // 2; runs = [online_run(e) for e in tqdm(dr, desc='online runs')]
drifted, control = runs[:half], runs[half:]
det = lambda rs: [cusum(r.edge.values) for r in rs]; d_idx, c_idx = det(drifted), det(control)
delay = [drifted[i].t.values[k] - 100 for i, k in enumerate(d_idx) if k is not None]
post = lambda rs, col: np.mean([r[r.t >= 100][col].mean() for r in rs])
print(f'CUSUM alarms: drifted {sum(k is not None for k in d_idx)}/{half}, control {sum(k is not None for k in c_idx)}/{half};  median detection delay vs drift time: {np.median(delay) if delay else float("nan"):.0f} h (negative = alarm before drift: weather onset also moves this edge)')
print(f'post-drift prequential NLL (drifted runs): frozen prior {post(drifted, "nll_frozen"):.4f} -> online {post(drifted, "nll_online"):.4f}   (control: {post(control, "nll_frozen"):.4f} -> {post(control, "nll_online"):.4f})')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); tt = drifted[0].t.values
for rs, c, lab in [(drifted, 'tab:red', 'drifted'), (control, 'tab:blue', 'control')]:
    E = np.array([r.edge.values for r in rs]); ax[0].plot(tt, E.mean(0), c=c, label=lab); ax[0].fill_between(tt, np.percentile(E, 25, 0), np.percentile(E, 75, 0), color=c, alpha=.15)
    R = np.array([r.rho.values for r in rs]); ax[2].plot(tt, R.mean(0), c=c, label=lab)
ax[0].axvline(100, c='k', ls='--'); ax[0].set_title('HYD→TRF learned influence (online)'); ax[0].legend(); ax[0].set_xlabel('hour')
gap = np.array([r.nll_frozen.values - r.nll_online.values for r in drifted]).mean(0); gap_c = np.array([r.nll_frozen.values - r.nll_online.values for r in control]).mean(0)
ax[1].plot(tt, gap, c='tab:red', label='drifted'); ax[1].plot(tt, gap_c, c='tab:blue', label='control'); ax[1].axhline(0, c='k', lw=.8); ax[1].axvline(100, c='k', ls='--'); ax[1].set_title('prequential NLL gain of online vs frozen prior'); ax[1].legend(); ax[1].set_xlabel('hour')
ax[2].axhline(1, c='r', ls='--'); ax[2].set_title('ρ over time (online)'); ax[2].set_xlabel('hour'); savefig('15_online_drift')

In [ ]:
# ---------------- save artifacts + report
rho, _ = W.branching_ratio(PRIOR['A'])
np.savez(f'{OUT}/hawkes_prior.npz', A=PRIOR['A'], mu=PRIOR['mu'], s=PRIOR['s'], taus=W.TAUS, rho=rho, edge_thr=em['thr'], l1=best[0], lg=best[1])
rep = dict(selected=dict(l1=best[0], lg=best[1]), val_nll=float(results[best]['hist'].val_nll.iloc[-1]), const_rate_val_nll=base_val, rho=rho, edge_metrics={k: float(v) for k, v in em.items()},
           rho_incl_same_layer=rho_total, baselines=TAB.to_dict('records'), lag_mae_h=lag_mae, heldout=dict(nll_zero_shot=nll_zs, nll_const=base_ho, scratch_auroc=em_scr['auroc'], lagcorr_auroc=lc[0]),
           rho_by_hazard=rho_h, drift=dict(alarms_drifted=int(sum(k is not None for k in d_idx)), alarms_control=int(sum(k is not None for k in c_idx)), n_pairs=half, median_delay_h=float(np.median(delay)) if delay else None,
           post_nll_frozen=float(post(drifted, 'nll_frozen')), post_nll_online=float(post(drifted, 'nll_online'))))
json.dump(rep, open(f'{OUT}/hawkes_report.json', 'w'), indent=1); CARDS.to_csv(f'{OUT}/hawkes_edge_cards.csv', index=False); print(json.dumps(rep, indent=1)[:1500])